## 0 · Use this checkout, not the installed package

Run this before anything else, and re-run it after a kernel restart.


In [ ]:
# Setup — run this FIRST.
#
# Order matters. Jupyter imports whatever `shipit_agent` is installed in the
# kernel, which lags this checkout: `include_server_in_tool_names` exists in
# the repo and not in the released package, so RemoteMCPServer(...) raises
# TypeError on an argument that is genuinely there. Putting the repo ahead of
# site-packages BEFORE the first import is the whole fix — adjusting sys.path
# afterwards is too late, the wrong module is already cached.
import sys
import pathlib

repo = pathlib.Path.cwd().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import shipit_agent
from shipit_agent import runtime as shipit_runtime
print(f"shipit-agent {shipit_agent.__version__}")
print(f"from {shipit_agent.__file__}")
assert str(repo) in shipit_agent.__file__, (
    "still importing the installed copy — restart the kernel and run this cell first"
)
assert getattr(shipit_runtime, 'EXPLICIT_MCP_ROUTING', False), (
    'stale runtime is cached — restart the kernel, then Run All'
)

# Agent + MCP Token and Streaming Audit

End-to-end validation of the main `Agent` with Gemma 4 on Bedrock Mantle and the public DeepWiki MCP. This notebook prints the complete tool/event stream, measures token usage, verifies canonical output retention, and exercises failure and large-output paths.

## What this validates

- Streamable HTTP MCP discovery and calls
- Per-call MCP `_meta` without leaking it into tool arguments
- Every lifecycle event and every generated text chunk
- Canonical versus model-visible tool output sizes
- Prompt, completion, cache, and total token usage
- Bounded live chunks for very large tool output
- Structured `run_failed` events
- Safe skill activation without unrelated tool injection
- Automatic progressive discovery with the full builtin catalog
- Multiple MCP servers plus local tools in one evidence-driven task
- Proof that simple chat executes no tools or MCP capabilities

In [ ]:
import importlib.util
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

from shipit_agent import Agent, FunctionTool, StreamRenderer, format_event_line
from shipit_agent.llms import LLMResponse, LiteLLMChatLLM
from shipit_agent.mcp import MCPStreamableHTTPTransport, RemoteMCPServer
from shipit_agent.models import ToolCall
from shipit_agent.tools import ToolContext

print('Python:', sys.version.split()[0])
print('Workspace:', Path.cwd())

## Load the supplied Gemma Mantle provider

The path and model are environment-overridable. No API key or credential is embedded in this notebook.

In [ ]:
DEFAULT_PROVIDER = (
    '/Users/rahulraj/Documents/MYWORK/AFTDRK/CACHE/DRK_CACHE_BACK/'
    'drk_cache/llm/bedrock_mantle_provider.py'
)
PROVIDER = Path(os.getenv('SHIPIT_MANTLE_PROVIDER', DEFAULT_PROVIDER))
MODEL = os.getenv('SHIPIT_AUDIT_MODEL', 'bedrock-mantle/google.gemma-4-26b-a4b')
assert PROVIDER.exists(), f'Provider not found: {PROVIDER}'

spec = importlib.util.spec_from_file_location('bedrock_mantle_provider', PROVIDER)
module = importlib.util.module_from_spec(spec)
sys.modules['bedrock_mantle_provider'] = module
spec.loader.exec_module(module)
module.ensure_registered()

def new_llm():
    return LiteLLMChatLLM(model=MODEL)

print('Provider:', PROVIDER)
print('Model:', MODEL)

## Connect DeepWiki MCP

DeepWiki is public and needs no authentication. The metadata resolver demonstrates generic trace context attached through MCP `_meta`.

In [ ]:
DEEPWIKI_URL = 'https://mcp.deepwiki.com/mcp'

def new_deepwiki(*, namespaced=False):
    return RemoteMCPServer(
        name='deepwiki',
        transport=MCPStreamableHTTPTransport(DEEPWIKI_URL, timeout=120),
        include_server_in_tool_names=namespaced,
        tool_meta_resolver=lambda context, tool, arguments: {
            'trace_id': context.metadata.get('trace_id', 'not-set'),
            'client': 'shipit-notebook-audit',
        },
    )

server = new_deepwiki()
started = time.perf_counter()
try:
    discovered = server.discover_tools()
    print('Server:', server.server_info)
    print('Protocol:', server.protocol_version)
    print('Discovery seconds:', round(time.perf_counter() - started, 2))
    for tool in discovered:
        print('-', tool.name, json.dumps(tool.input_schema))
finally:
    server.close()

## Direct MCP baseline

This isolates remote retrieval latency and response size from model overhead.

## Build the main Agent

Only DeepWiki tools are exposed for this focused test. Complete results remain caller-visible; only the copy entering future model turns is subject to context budgets.

In [ ]:

QUESTION = (
    'Concisely explain retry eligibility, backoff, and streaming cleanup. '
    'Name the core classes and methods and give two failure tests.'
)
def new_agent():
    return Agent(
        llm=new_llm(),
        # Progress summaries stay runtime-derived so this audit adds no
        # secondary paid model completions.
        mcps=[new_deepwiki(namespaced=True)],
        prompt=(
            'You are a precise repository research agent. Call DeepWiki before '
            'repository claims. Use declared arguments only. Ground the answer '
            'in canonical tool output.'
        ),
        name='gemma-deepwiki-notebook-audit',
        metadata={'trace_id': 'notebook-live-run'},
        max_iterations=6,
        progress_summaries=True,
        parallel_tool_execution=True,
        max_tool_concurrency=3,
        max_tool_output_chars=16_000,
        max_tool_output_group_chars=48_000,
        persist_large_tool_outputs=True,
        project_root='.',
    )

agent = new_agent()
TASK = (
    'Use DeepWiki ask_question on openai/openai-python. ' + QUESTION +
    ' Call exactly one MCP tool unless it fails.'
)
selected = agent._selected_skills(TASK)
print('Selected skills:', [skill.id for skill in selected])
print('Effective max iterations:', agent._effective_max_iterations(selected))

## Complete raw event stream

The MCP body is printed once from `tool_output_delta`. Final text chunks print continuously as they arrive. Heavy canonical MCP metadata is intentionally not duplicated into every delta.

In [ ]:
events = []
renderer = StreamRenderer(style='plain')
try:
    for event in agent.stream(TASK):
        events.append(event)
        renderer.feed(event)
finally:
    renderer.close()

In [ ]:

import importlib.util
import json
import os
import subprocess
import tempfile
from collections import Counter
from types import SimpleNamespace

workspace_handle = tempfile.TemporaryDirectory(prefix='shipit-live-audit-')
workspace = Path(workspace_handle.name)
(workspace / 'calculator.py').write_text('def divide(total, count):\n    return total * count\n')
(workspace / 'test_calculator.py').write_text(
    'from calculator import divide\n\ndef test_divide():\n    assert divide(12, 3) == 4\n'
)


In [ ]:
from shipit_agent.prompts.default_agent_prompt import DEFAULT_AGENT_PROMPT
from shipit_agent.builtins import get_builtin_tool_map
LANGUAGE_CONTRACT = (
    '\n\n## Output language\nUse English only for public progress and final answers. '
    'Do not translate or repeat progress in another language.'
)
AUDIT_AGENT_PROMPT = DEFAULT_AGENT_PROMPT + LANGUAGE_CONTRACT


In [ ]:
tool_map = get_builtin_tool_map(llm=new_llm(), project_root=str(workspace))
coding_tools = [tool_map[name] for name in ('read_file', 'grep_files', 'edit_file', 'bash')]
coding_agent = Agent(
    llm=new_llm(),
    mcps=[new_deepwiki(namespaced=True)],
    prompt=AUDIT_AGENT_PROMPT,
    tools=coding_tools,
    project_root=str(workspace),
    permission_mode='bypass',
    tool_context_mode='full',
    auto_use_skills=False,
    name='gemma-deepwiki-notebook-audit',
    metadata={'trace_id': 'notebook-live-run'},
    progress_summaries=True,
    parallel_tool_execution=True,
    max_tool_concurrency=3,
    max_tool_output_chars=16_000,
    max_tool_output_group_chars=48_000,
    persist_large_tool_outputs=True,
    max_iterations=14,
)

In [ ]:
coding_events = []
renderer = StreamRenderer(style='plain')
try:
    for event in coding_agent.stream(
        'Inspect calculator.py and its test. Fix the implementation bug with '
        'edit_file, then run pytest with bash. Do not change the test and do not '
        'finish until pytest passes.'
    ):
        coding_events.append(event)
        renderer.feed(event)
finally:
    renderer.close()

In [ ]:
TASK = (
    'Use DeepWiki ask_question on openai/openai-python. ' + QUESTION +
    ' Call exactly one MCP tool unless it fails.'
)

# This section performs one measured agent run below. No decision_llm is
# configured, so progress_summaries adds no paid model completions.
assert agent.decision_llm is None
print('Progress mode: runtime-derived, zero extra LLM calls')


In [ ]:
# Raw events are captured and replayed after the measured stream below.
print('Run the next live-stream cell, then inspect the raw-event replay.')


In [ ]:
QUESTION = (
    'Concisely explain retry eligibility, backoff, and streaming cleanup. '
    'Name the core classes and methods and give two failure tests.'
)
server = new_deepwiki()
try:
    ask = next(tool for tool in server.discover_tools() if tool.name == 'ask_question')
    started = time.perf_counter()
    direct = ask.run(
        ToolContext(prompt=QUESTION, metadata={'trace_id': 'direct-baseline'}),
        repoName='openai/openai-python',
        question=QUESTION,
    )
    direct_seconds = time.perf_counter() - started
    print({
        'ok': direct.metadata.get('ok'),
        'seconds': round(direct_seconds, 2),
        'characters': len(direct.text),
        'words': len(direct.text.split()),
    })
    print(direct.text)
finally:
    server.close()

In [ ]:
event_counts = Counter()
completed_payload = {}
tool_telemetry = []
events = []
in_text = False
started = time.perf_counter()

for event in agent.stream(TASK):
    # print('event', event)
    events.append(event)
    event_counts[event.type] += 1
    payload = event.payload
    if event.type == 'text_delta':
        if not in_text:
            print('\ntext_delta stream:')
            in_text = True
        print(payload.get('chunk', ''), end='', flush=True)
        continue
    if in_text:
        print('\n[end text_delta stream]')
        in_text = False
    if event.type == 'tool_output_delta':
        chunk = str(payload.get('chunk', ''))
        print('tool_output_delta', {
            'tool': payload.get('tool'),
            'sequence': payload.get('sequence'),
            'characters': len(chunk),
            'metadata': payload.get('chunk_metadata'),
        })
        print(chunk)
    elif event.type == 'tool_completed':
        telemetry = {
            'tool': payload.get('tool'),
            'output_chars': payload.get('output_chars'),
            'model_output_chars': payload.get('model_output_chars'),
            'model_output_reduced': payload.get('model_output_reduced'),
            'metadata': payload.get('metadata'),
        }
        tool_telemetry.append(telemetry)
        print('tool_completed', json.dumps(telemetry, default=str))
    elif event.type == 'run_completed':
        completed_payload = dict(payload)
        print('run_completed', {
            'usage': payload.get('usage'),
            'output_chars': len(str(payload.get('output', ''))),
            'cancelled': payload.get('cancelled'),
        })
    else:
        display_line = format_event_line(event)
        if display_line:
            print(display_line)
        else:
            safe = {
                key: value for key, value in payload.items()
                if key not in {'output', 'content', 'chunk'}
            }
            print(event.type, json.dumps(safe, ensure_ascii=False, default=str))

if in_text:
    print('\n[end text_delta stream]')
agent_seconds = time.perf_counter() - started

In [ ]:
# Full old-style diagnostic output: payloads, tool results, telemetry, timestamps.
# This reprints the captured run; it does not call the model or MCP again.
for event in events:
    print('event', repr(event))

## Measured run report

In [ ]:
# Human-facing progress only. The complete low-level trace remains in `events`.
progress_types = {
    'skills_selected', 'agent_decision', 'tool_called', 'agent_observation',
    'tool_failed', 'run_failed', 'run_completed',
}
def progress_line(event):
    return (
        format_event_line(event)
        or f'{event.type:<20} {event.display_message}'.rstrip()
    )

progress_transcript = [
    progress_line(event) for event in events if event.type in progress_types
]
print('\n'.join(progress_transcript))
print('\nRaw event counts:', dict(event_counts))

In [ ]:
usage = completed_payload.get('usage', {})
token_report = {
    'prompt_tokens': usage.get('prompt_tokens', 0),
    'completion_tokens': usage.get('completion_tokens', 0),
    'total_tokens': usage.get('total_tokens', 0),
    'cache_read_input_tokens': usage.get('cache_read_input_tokens', 0),
    'cache_creation_input_tokens': usage.get('cache_creation_input_tokens', 0),
}
run_report = {
    'wall_seconds': round(agent_seconds, 2),
    'event_counts': dict(event_counts),
    'usage': token_report,
    'final_output_chars': len(str(completed_payload.get('output', ''))),
    'tool_telemetry': tool_telemetry,
}
called_tools = [
    str(event.payload.get('tool', '')) for event in events
    if event.type == 'tool_called'
]
mcp_calls = [name for name in called_tools if name.startswith('deepwiki__')]
unmet = [event for event in events if event.type == 'requirements_unmet']
compacted = [event for event in events if event.type == 'model_output_compacted']
final_text = str(completed_payload.get('output', ''))
assert mcp_calls == ['deepwiki__ask_question'], called_tools
assert 'tool_search' not in called_tools, called_tools
assert not unmet, [event.payload for event in unmet]
assert not compacted, [event.payload for event in compacted]
assert len(final_text) >= 300, final_text
print(json.dumps(run_report, indent=2, default=str))
print('\nFINAL ANSWER\n')
print(completed_payload.get('output', ''))

## Large-output streaming stress test

This local deterministic test proves that a 100,000-character result stays complete while live deltas remain bounded and the model-visible copy is capped.

In [ ]:
class OneToolThenAnswer:
    def __init__(self):
        self.calls = 0

    def complete(self, **kwargs):
        self.calls += 1
        if self.calls == 1:
            return LLMResponse(
                content='',
                tool_calls=[ToolCall(name='large_result', arguments={})],
                usage={'prompt_tokens': 10, 'completion_tokens': 2, 'total_tokens': 12},
            )
        return LLMResponse(
            content='Large result processed.',
            usage={'prompt_tokens': 20, 'completion_tokens': 4, 'total_tokens': 24},
        )

large_text = '0123456789' * 10_000
large_agent = Agent(
    llm=OneToolThenAnswer(),
    tools=[FunctionTool.from_callable(lambda: large_text, name='large_result')],
    auto_use_skills=False,
    max_tool_output_chars=4_000,
    max_tool_output_group_chars=4_000,
)
large_events = list(large_agent.stream('Run the large result tool.'))
large_deltas = [e.payload['chunk'] for e in large_events if e.type == 'tool_output_delta']
large_completed = next(e for e in large_events if e.type == 'tool_completed')
print({
    'canonical_chars': large_completed.payload['output_chars'],
    'model_chars': large_completed.payload['model_output_chars'],
    'model_output_reduced': large_completed.payload['model_output_reduced'],
    'delta_count': len(large_deltas),
    'largest_delta': max(map(len, large_deltas)),
    'reassembled_complete': ''.join(large_deltas) == large_text,
})

## Structured failure stream

Provider failures still raise to the caller, but consumers receive a terminal `run_failed` event first.

In [ ]:
class BrokenLLM:
    def complete(self, **kwargs):
        raise ConnectionError('simulated provider outage')

failure_events = []
stream = Agent(llm=BrokenLLM(), auto_use_skills=False).stream('test failure')
try:
    while True:
        event = next(stream)
        failure_events.append(event)
        print(event.type, event.payload)
except StopIteration:
    pass
except ConnectionError as exc:
    print('Expected exception:', exc)

failed = [event for event in failure_events if event.type == 'run_failed']
assert len(failed) == 1
assert failed[0].payload['retryable'] is True

## Skill activation audit

Fuzzy catalog search is explicit. Runtime auto-activation only uses authored trigger phrases, preventing unrelated prompts and tool bundles from inflating every model turn.

In [ ]:
skill_agent = Agent(llm=OneToolThenAnswer())
mcp_prompt = 'Use DeepWiki MCP to analyze retry behavior in a repository.'
trigger_prompt = 'Please debug this production bug and plan this feature.'
print('MCP prompt skills:', [s.id for s in skill_agent._selected_skills(mcp_prompt)])
print('Authored-trigger skills:', [s.id for s in skill_agent._selected_skills(trigger_prompt)])
print('Explicit fuzzy search:', [s.id for s in skill_agent.search_skills('database')[:5]])
assert skill_agent._selected_skills(mcp_prompt) == []
assert skill_agent._selected_skills(trigger_prompt)

## Progressive super-agent live matrix

This section uses the real Bedrock LLM with the **full builtin catalog**, the live DeepWiki MCP, a deterministic operations MCP, and custom local tools. It tests no-tool chat, one-domain investigation, and a deep cross-domain investigation. MCP `initialize` / `tools/list` makes capabilities searchable; `tool_called` events show which capabilities were actually executed.

In [ ]:
from shipit_agent import MCPServer, MCPTool

INCIDENTS = {
    'INC-1042': {
        'service': 'payments-api',
        'symptom': 'retry storm after upstream 429 responses',
        'error_rate_pct': 8.7,
        'p95_ms': 2410,
        'deploy': '2026.08.10-rc3',
        'notes': 'stream cancellations leave work pending for 30-45 seconds',
    },
    'INC-1038': {
        'service': 'search-api',
        'symptom': 'connection pool exhaustion',
        'error_rate_pct': 2.1,
        'p95_ms': 890,
        'deploy': '2026.08.09',
        'notes': 'resolved by bounded concurrency',
    },
}

def new_ops_mcp():
    def lookup(context=None, incident_id='', **_):
        record = INCIDENTS.get(incident_id)
        return json.dumps(record or {'error': 'incident not found'}, indent=2)

    def compare(context=None, incident_ids=None, **_):
        ids = list(incident_ids or [])
        return json.dumps({key: INCIDENTS.get(key) for key in ids}, indent=2)

    return MCPServer(name='operations').register_many([
        MCPTool(
            name='ops_lookup_incident',
            description='Look up one production incident by exact incident id.',
            handler=lookup,
            input_schema={
                'type': 'object',
                'properties': {'incident_id': {'type': 'string'}},
                'required': ['incident_id'],
            },
        ),
        MCPTool(
            name='ops_compare_incidents',
            description='Compare several production incidents.',
            handler=compare,
            input_schema={
                'type': 'object',
                'properties': {
                    'incident_ids': {'type': 'array', 'items': {'type': 'string'}},
                },
                'required': ['incident_ids'],
            },
        ),
    ])

def calculate_error_budget(error_rate_pct: float, window_minutes: int = 60):
    """Calculate failed minutes and remaining 99.9% SLO budget."""
    allowed = window_minutes * 0.001
    failed = window_minutes * (error_rate_pct / 100.0)
    return {
        'window_minutes': window_minutes,
        'failed_minutes': round(failed, 3),
        'allowed_minutes': round(allowed, 3),
        'budget_remaining_minutes': round(allowed - failed, 3),
        'slo_breached': failed > allowed,
    }

def release_risk(deploy: str, error_rate_pct: float):
    """Return a deterministic release-risk classification."""
    score = min(100, round(error_rate_pct * 9 + (15 if 'rc' in deploy else 0)))
    return {
        'deploy': deploy,
        'risk_score': score,
        'risk': 'high' if score >= 60 else 'medium',
    }

LOCAL_AUDIT_TOOLS = [
    FunctionTool.from_callable(calculate_error_budget, name='calculate_error_budget'),
    FunctionTool.from_callable(release_risk, name='release_risk'),
]

def new_progressive_super_agent():
    return Agent.with_builtins(
        llm=new_llm(),
        tools=LOCAL_AUDIT_TOOLS,
        mcps=[new_deepwiki(namespaced=True), new_ops_mcp()],
        prompt=(
            'You are a production incident investigator. Use tools only when evidence is '
            'needed. For repository claims, use DeepWiki or inspect this checkout. For incident '
            'facts, use the operations MCP. Never call unrelated SaaS connectors. Distinguish '
            'observed evidence from inference and finish with concrete verification steps.'
        ),
        name='progressive-live-super-agent',
        metadata={'trace_id': 'notebook-78-progressive-live'},
        project_root=str(repo),
        tool_context_mode='auto',
        tool_context_threshold_chars=12_000,
        auto_use_skills=False,
        max_iterations=18,
        parallel_tool_execution=True,
        max_tool_concurrency=4,
        max_tool_output_chars=12_000,
        max_tool_output_group_chars=32_000,
        persist_large_tool_outputs=True,
    )

print('Local tools:', [tool.name for tool in LOCAL_AUDIT_TOOLS])
print('Operations MCP:', [tool.name for tool in new_ops_mcp().discover_tools()])

In [ ]:
DISCOVERY_TOOLS = {
    'tool_search', 'call_tool', 'describe_binding', 'execute_code', 'todo'
}

def run_live_case(label, prompt):
    print(f'\n{"=" * 24} {label} {"=" * 24}')
    case_agent = new_progressive_super_agent()
    case_events = []
    started = time.perf_counter()
    for event in case_agent.stream(prompt):
        case_events.append(event)
        if event.type in {
            'skills_selected', 'mcp_attached', 'tool_called', 'tool_denied',
            'tool_completed', 'context_compacted', 'run_failed',
        }:
            print(format_event_line(event) or f'{event.type}: {event.payload}')
        elif event.type == 'text_delta':
            print(event.payload.get('chunk', ''), end='', flush=True)
    print()
    completed = next(
        event for event in reversed(case_events) if event.type == 'run_completed'
    )
    called = [
        event.payload.get('tool') for event in case_events
        if event.type == 'tool_called' and event.payload.get('tool')
    ]
    actual = [name for name in called if name not in DISCOVERY_TOOLS]
    discovered_bindings = [
        str(event.payload.get('arguments', {}).get('name', '')).lower()
        for event in case_events
        if event.type == 'tool_called'
        and event.payload.get('tool') == 'describe_binding'
    ]
    report = {
        'label': label,
        'seconds': round(time.perf_counter() - started, 2),
        'called_tools': called,
        'actual_capabilities': actual,
        'discovered_bindings': discovered_bindings,
        'tool_context': completed.payload.get('tool_context', {}),
        'usage': completed.payload.get('usage', {}),
        'output': completed.payload.get('output', ''),
    }
    print('\nCASE REPORT')
    print(json.dumps({
        key: value for key, value in report.items() if key != 'output'
    }, indent=2))
    print('\nFINAL ANSWER\n', report['output'])
    return report, case_events


### Case A — ordinary chat must call nothing

The full catalog and two MCP servers are attached, but this prompt needs no external evidence.

In [ ]:
simple_report, simple_events = run_live_case(
    'NO-TOOL CHAT',
    'Say hello and explain in one sentence what an idempotent operation is. Do not use tools.',
)
assert simple_report['called_tools'] == [], simple_report['called_tools']
assert simple_report['tool_context']['hidden'] > 0


### Case B — focused incident question

This should call the operations MCP and only the calculations needed for the answer.

In [ ]:
incident_report, incident_events = run_live_case(
    'FOCUSED INCIDENT',
    (
        'Investigate INC-1042. You MUST execute ops_lookup_incident, '
        'calculate_error_budget, and release_risk with their exact schemas. After discovering '
        'the schemas, batch all three calls in one execute_code block and do not calculate '
        'their outputs yourself. Recommend the single safest immediate action. Do not search '
        'the web or call repository tools. All env binding arguments must be keyword arguments.'
    ),
)
assert 'ops_lookup_incident' in incident_report['actual_capabilities'], incident_report
assert {
    'ops_lookup_incident', 'calculate_error_budget', 'release_risk'
} <= set(incident_report['discovered_bindings']), incident_report


### Case C — deep cross-domain investigation

The answer must combine production facts, live external repository research, and this checkout's retry implementation. Independent reads may run in parallel; unrelated connectors must remain dormant.

In [ ]:
DEEP_LIVE_QUESTION = '''
Investigate whether retry and streaming-cleanup behavior could explain INC-1042.

Build an evidence-backed incident review using exactly these evidence domains:
1. Retrieve INC-1042 from the operations MCP.
2. Use DeepWiki on openai/openai-python to identify retry eligibility, exponential backoff,
   jitter, Retry-After handling, streaming cancellation, and cleanup classes/methods.
3. Inspect this local shipit_agent checkout with grep_files/read_file for RetryPolicy, the LLM
   retry loop, tool retry loop, stream cleanup, and MCP transport closing behavior.

Then produce an observed-evidence table, a causal analysis separating confirmed facts from
inference, an openai-python versus shipit_agent comparison, and three prioritized fixes with
concrete tests and rollback signals.

Do not call Gmail, Slack, Stripe, calendars, CRM, design, or unrelated connectors.
Do not claim that an implementation detail exists unless a tool result supports it.
'''
deep_report, deep_events = run_live_case(
    'DEEP CROSS-DOMAIN INVESTIGATION',
    DEEP_LIVE_QUESTION,
)
deep_actual = set(deep_report['actual_capabilities'])
assert 'ops_lookup_incident' in deep_actual, deep_actual
assert any(name.startswith('deepwiki__') for name in deep_actual), deep_actual
deep_local_coverage = sorted(
    deep_actual & {'grep_files', 'read_file', 'bash', 'glob_files'}
)
print('Local checkout tools actually used:', deep_local_coverage or 'none (model stopped early)')
for forbidden in ('gmail', 'slack', 'stripe', 'google_calendar', 'salesforce', 'figma'):
    assert forbidden not in deep_actual, (forbidden, deep_actual)


In [ ]:
live_matrix = [simple_report, incident_report, deep_report]
matrix_summary = [
    {
        'case': report['label'],
        'seconds': report['seconds'],
        'actual_capabilities': report['actual_capabilities'],
        'registered': report['tool_context'].get('registered'),
        'exposed': report['tool_context'].get('exposed'),
        'hidden': report['tool_context'].get('hidden'),
        'prompt_tokens': report['usage'].get('prompt_tokens', 0),
        'completion_tokens': report['usage'].get('completion_tokens', 0),
    }
    for report in live_matrix
]
print(json.dumps(matrix_summary, indent=2))
assert simple_report['actual_capabilities'] == []
assert all(row['hidden'] > row['exposed'] for row in matrix_summary)


## Final assertions

In [ ]:
checks = {
    'deepwiki_direct_ok': direct.metadata.get('ok') is True,
    'agent_completed': bool(completed_payload.get('output')),
    'one_underlying_mcp_call': len([
        event for event in events
        if event.type == 'tool_called'
        and str(event.payload.get('tool', '')).startswith('deepwiki__')
    ]) == 1,
    'usage_reported': usage.get('total_tokens', 0) > 0,
    'canonical_result_complete': all(t['output_chars'] > 0 for t in tool_telemetry),
    'large_result_complete': ''.join(large_deltas) == large_text,
    'large_model_copy_reduced': large_completed.payload['model_output_reduced'],
    'failure_event_emitted': len(failed) == 1,
    'no_unrelated_skill_injection': skill_agent._selected_skills(mcp_prompt) == [],
    'simple_chat_used_no_tools': simple_report['actual_capabilities'] == [],
    'focused_case_used_ops_mcp': (
        'ops_lookup_incident' in incident_report['actual_capabilities']
    ),
    'deep_case_used_both_mcp_domains': (
        'ops_lookup_incident' in deep_report['actual_capabilities']
        and any(
            name.startswith('deepwiki__')
            for name in deep_report['actual_capabilities']
        )
    ),
    'progressive_catalog_is_mostly_hidden': all(
        row['hidden'] > row['exposed'] for row in matrix_summary
    ),
}
for name, passed in checks.items():
    print(f'{"PASS" if passed else "FAIL"}: {name}')
assert all(checks.values()), checks
print('\nAll agent, MCP, streaming, token, and failure checks passed.')